# Sleep Apnea Detection - Exploratory Data Analysis

Analyzing HR and SpO2 signals from 50 patients for sleep apnea detection.
- **Inputs**: Heart Rate (bpm) + SpO2 (%) only
- **Labels**: Binary -- both hypo and osa = apnea
- **Goal**: Understand signal patterns, class balance, data quality

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_patient, load_annotations, PATIENT_IDS, DATA_DIR

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Dataset Overview

In [ ]:
patient_stats = []
for pid in PATIENT_IDS:
    try:
        p = load_patient(pid)
        ann = p['annotations']
        signals = p['signals']
        n_events = len(ann['events'])
        n_hypo = sum(1 for e in ann['events'] if e['event_type'] == 'hypo')
        n_osa = sum(1 for e in ann['events'] if e['event_type'] == 'osa')
        duration_hrs = (signals['timestamp_sec'].iloc[-1] - signals['timestamp_sec'].iloc[0]) / 3600
        ahi = n_events / duration_hrs if duration_hrs > 0 else 0
        hr_missing = signals['HR'].isna().sum() / len(signals) * 100
        spo2_missing = signals['SpO2'].isna().sum() / len(signals) * 100
        
        patient_stats.append({
            'patient_id': pid,
            'n_samples': len(signals),
            'duration_hrs': round(duration_hrs, 2),
            'n_events': n_events,
            'n_hypo': n_hypo,
            'n_osa': n_osa,
            'ahi': round(ahi, 1),
            'hr_mean': round(signals['HR'].mean(), 1),
            'spo2_mean': round(signals['SpO2'].mean(), 1),
            'hr_missing_pct': round(hr_missing, 1),
            'spo2_missing_pct': round(spo2_missing, 1),
        })
    except Exception as e:
        print(f'Patient {pid}: {e}')

stats_df = pd.DataFrame(patient_stats)
print(f'Successfully loaded {len(stats_df)} patients')
stats_df.head(10)

In [ ]:
print('=== Dataset Summary ===')
print(f'Patients: {len(stats_df)}')
print(f'Recording duration: {stats_df["duration_hrs"].mean():.1f} +/- {stats_df["duration_hrs"].std():.1f} hours')
print(f'\nApnea Events per patient:')
print(f'  Total:  {stats_df["n_events"].mean():.0f} +/- {stats_df["n_events"].std():.0f} (range: {stats_df["n_events"].min()}-{stats_df["n_events"].max()})')
print(f'  Hypo:   {stats_df["n_hypo"].mean():.0f} +/- {stats_df["n_hypo"].std():.0f}')
print(f'  OSA:    {stats_df["n_osa"].mean():.0f} +/- {stats_df["n_osa"].std():.0f}')
print(f'\nAHI (Apnea-Hypopnea Index):')
print(f'  Mean: {stats_df["ahi"].mean():.1f}, Median: {stats_df["ahi"].median():.1f}, Range: {stats_df["ahi"].min()}-{stats_df["ahi"].max()}')
print(f'\nMissing data:')
print(f'  HR missing:   {stats_df["hr_missing_pct"].mean():.1f}% mean (max: {stats_df["hr_missing_pct"].max():.1f}%)')
print(f'  SpO2 missing: {stats_df["spo2_missing_pct"].mean():.1f}% mean (max: {stats_df["spo2_missing_pct"].max():.1f}%)')

## 2. AHI Distribution Across Patients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AHI distribution
axes[0].bar(stats_df['patient_id'], stats_df['ahi'], color='steelblue')
axes[0].axhline(y=5, color='green', linestyle='--', alpha=0.7, label='Mild (5)')
axes[0].axhline(y=15, color='orange', linestyle='--', alpha=0.7, label='Moderate (15)')
axes[0].axhline(y=30, color='red', linestyle='--', alpha=0.7, label='Severe (30)')
axes[0].set_xlabel('Patient ID')
axes[0].set_ylabel('AHI (events/hour)')
axes[0].set_title('AHI Distribution Across Patients')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=90, labelsize=7)

# Severity classification
severity = pd.cut(stats_df['ahi'], bins=[0, 5, 15, 30, 999],
                   labels=['Normal (<5)', 'Mild (5-15)', 'Moderate (15-30)', 'Severe (>30)'])
severity_counts = severity.value_counts().sort_index()
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
axes[1].pie(severity_counts, labels=severity_counts.index, autopct='%1.0f%%',
            colors=colors, startangle=90)
axes[1].set_title('Sleep Apnea Severity Distribution')

plt.tight_layout()
plt.savefig('../outputs/eda_ahi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Event Type Distribution (Hypopnea vs OSA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar chart
x = range(len(stats_df))
axes[0].bar(x, stats_df['n_hypo'], label='Hypopnea', color='#3498db')
axes[0].bar(x, stats_df['n_osa'], bottom=stats_df['n_hypo'], label='OSA', color='#e74c3c')
axes[0].set_xlabel('Patient ID')
axes[0].set_ylabel('Number of Events')
axes[0].set_title('Apnea Events by Type per Patient')
axes[0].set_xticks(x)
axes[0].set_xticklabels(stats_df['patient_id'], rotation=90, fontsize=7)
axes[0].legend()

# Overall pie chart
total_hypo = stats_df['n_hypo'].sum()
total_osa = stats_df['n_osa'].sum()
axes[1].pie([total_hypo, total_osa], labels=[f'Hypopnea ({total_hypo})', f'OSA ({total_osa})'],
            autopct='%1.1f%%', colors=['#3498db', '#e74c3c'], startangle=90)
axes[1].set_title('Overall Event Type Distribution')

plt.tight_layout()
plt.savefig('../outputs/eda_event_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(stats_df))
width = 0.35
ax.bar(x - width/2, stats_df['hr_missing_pct'], width, label='HR Missing %', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, stats_df['spo2_missing_pct'], width, label='SpO2 Missing %', color='#3498db', alpha=0.8)
ax.axhline(y=20, color='black', linestyle='--', alpha=0.5, label='20% threshold')
ax.set_xlabel('Patient ID')
ax.set_ylabel('Missing Data %')
ax.set_title('Missing Data by Patient')
ax.set_xticks(x)
ax.set_xticklabels(stats_df['patient_id'], rotation=90, fontsize=7)
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/eda_missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Signal Patterns: Apnea vs Normal

Visualize HR and SpO2 during apnea events vs normal segments for a few patients.

In [ ]:
# Pick 3 patients with different severity levels
sample_patients = ['01', '20', '35']

for pid in sample_patients:
    p = load_patient(pid)
    signals = p['signals']
    events = p['annotations']['events']
    
    t = signals['timestamp_sec'].values
    t_relative = (t - t[0]) / 3600  # hours from start
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    
    # HR plot
    axes[0].plot(t_relative, signals['HR'].values, color='#e74c3c', linewidth=0.3, alpha=0.7)
    axes[0].set_ylabel('Heart Rate (bpm)')
    axes[0].set_title(f'Patient {pid} - HR and SpO2 with Apnea Events ({len(events)} events)')
    
    # SpO2 plot
    axes[1].plot(t_relative, signals['SpO2'].values, color='#3498db', linewidth=0.3, alpha=0.7)
    axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='SpO2=90%')
    axes[1].set_ylabel('SpO2 (%)')
    axes[1].set_xlabel('Time (hours from start)')
    axes[1].legend()
    
    # Shade apnea events
    for ev in events:
        ev_start = (ev['event_start'] - t[0]) / 3600
        ev_end = (ev['event_end'] - t[0]) / 3600
        for ax in axes:
            ax.axvspan(ev_start, ev_end, alpha=0.15, color='red')
    
    plt.tight_layout()
    plt.savefig(f'../outputs/eda_signals_patient_{pid}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Zoomed-in View: Single Apnea Event

Close-up of HR/SpO2 behavior around one apnea event to see the brady-tachy pattern and SpO2 desaturation.

In [ ]:
# Zoom into a ~5 minute window around an apnea cluster for patient 35
p = load_patient('35')
signals = p['signals']
events = p['annotations']['events']

# Pick events in the middle of the recording
mid_event = events[len(events) // 2]
center = mid_event['event_start']
zoom_start = center - 150
zoom_end = center + 150

mask = (signals['timestamp_sec'] >= zoom_start) & (signals['timestamp_sec'] <= zoom_end)
seg = signals.loc[mask].copy()
seg['t_sec'] = seg['timestamp_sec'] - zoom_start

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(seg['t_sec'], seg['HR'], color='#e74c3c', linewidth=1)
axes[0].set_ylabel('Heart Rate (bpm)')
axes[0].set_title(f'Patient 35 - Zoomed View (~5 min window)')

axes[1].plot(seg['t_sec'], seg['SpO2'], color='#3498db', linewidth=1)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('SpO2 (%)')
axes[1].set_xlabel('Time (seconds from window start)')

for ev in events:
    if ev['event_start'] >= zoom_start and ev['event_start'] <= zoom_end:
        es = ev['event_start'] - zoom_start
        ee = ev['event_end'] - zoom_start
        for ax in axes:
            ax.axvspan(es, ee, alpha=0.2, color='red')
        axes[0].annotate(ev['event_type'], xy=(es, axes[0].get_ylim()[1]),
                         fontsize=8, color='darkred', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/eda_zoomed_apnea.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. HR and SpO2 Distribution: Apnea vs Normal Windows

In [ ]:
from preprocess import clean_signals
from windowing import create_windows
from features import extract_features, FEATURE_NAMES

# Extract features for a subset of patients to visualize distributions
sample_pids = ['01', '05', '10', '15', '20', '25', '30', '35', '40', '45', '50']
all_feats = []

for pid in sample_pids:
    p = load_patient(pid)
    p['signals'] = clean_signals(p['signals'])
    wins = create_windows(p)
    for w in wins:
        feats = extract_features(w)
        feats['label'] = 'Apnea' if w['label'] == 1 else 'Normal'
        feats['patient_id'] = pid
        all_feats.append(feats)

feat_df = pd.DataFrame(all_feats)
print(f'Total windows from sample: {len(feat_df)}')
print(feat_df['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

key_features = ['hr_mean', 'hr_rmssd', 'hr_range', 'spo2_mean', 'spo2_min', 'spo2_delta']
titles = ['Mean HR', 'HR RMSSD (HRV)', 'HR Range', 'Mean SpO2', 'Min SpO2', 'SpO2 Delta']

for idx, (feat, title) in enumerate(zip(key_features, titles)):
    ax = axes[idx // 3][idx % 3]
    for label, color in [('Normal', '#2ecc71'), ('Apnea', '#e74c3c')]:
        data = feat_df.loc[feat_df['label'] == label, feat]
        ax.hist(data, bins=40, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(title, fontsize=12)
    ax.legend()
    ax.set_xlabel(feat)

plt.suptitle('Feature Distributions: Apnea vs Normal Windows', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Correlation Heatmap

In [ ]:
numeric_feats = feat_df[FEATURE_NAMES + ['label']].copy()
numeric_feats['label_num'] = (numeric_feats['label'] == 'Apnea').astype(int)
numeric_feats = numeric_feats.drop('label', axis=1)

corr = numeric_feats.corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix (including label)', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Show features most correlated with the label
label_corr = corr['label_num'].drop('label_num').sort_values(key=abs, ascending=False)
print('\nFeature correlations with Apnea label:')
for feat, c in label_corr.items():
    print(f'  {feat:25s} {c:+.3f}')

## 9. Class Balance Analysis (Full Dataset)

In [ ]:
from preprocess import preprocess_all_patients
from windowing import create_all_windows

print('Loading all 50 patients for class balance analysis...')
all_patients = preprocess_all_patients()
all_windows = create_all_windows(all_patients)

total = len(all_windows)
apnea = sum(1 for w in all_windows if w['label'] == 1)
normal = total - apnea

print(f'\n=== Class Balance (Full Dataset) ===')
print(f'Total windows:  {total}')
print(f'Apnea windows:  {apnea} ({100*apnea/total:.1f}%)')
print(f'Normal windows: {normal} ({100*normal/total:.1f}%)')
print(f'Imbalance ratio: 1:{normal/apnea:.1f}' if apnea > 0 else 'No apnea windows')

In [ ]:
# Per-patient class balance
patient_balance = {}
for w in all_windows:
    pid = w['patient_id']
    if pid not in patient_balance:
        patient_balance[pid] = {'apnea': 0, 'normal': 0}
    if w['label'] == 1:
        patient_balance[pid]['apnea'] += 1
    else:
        patient_balance[pid]['normal'] += 1

balance_df = pd.DataFrame(patient_balance).T.reset_index()
balance_df.columns = ['patient_id', 'apnea', 'normal']
balance_df['total'] = balance_df['apnea'] + balance_df['normal']
balance_df['apnea_pct'] = (balance_df['apnea'] / balance_df['total'] * 100).round(1)
balance_df = balance_df.sort_values('patient_id')

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(balance_df))
ax.bar(x, balance_df['normal'], label='Normal', color='#2ecc71')
ax.bar(x, balance_df['apnea'], bottom=balance_df['normal'], label='Apnea', color='#e74c3c')
ax.set_xlabel('Patient ID')
ax.set_ylabel('Number of Windows')
ax.set_title('Class Balance per Patient (60s windows)')
ax.set_xticks(x)
ax.set_xticklabels(balance_df['patient_id'], rotation=90, fontsize=7)
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nApnea % per patient: min={balance_df["apnea_pct"].min():.1f}%, '
      f'max={balance_df["apnea_pct"].max():.1f}%, '
      f'mean={balance_df["apnea_pct"].mean():.1f}%')

## 10. Event Duration Distribution

In [ ]:
all_durations = []
for p in all_patients:
    for ev in p['annotations']['events']:
        all_durations.append({
            'duration': ev['event_duration'],
            'type': ev['event_type'],
        })

dur_df = pd.DataFrame(all_durations)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dur_df['duration'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(x=10, color='red', linestyle='--', label='10s threshold')
axes[0].set_xlabel('Event Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Apnea Event Duration Distribution')
axes[0].legend()

for etype, color in [('hypo', '#3498db'), ('osa', '#e74c3c')]:
    data = dur_df.loc[dur_df['type'] == etype, 'duration']
    axes[1].hist(data, bins=40, alpha=0.6, label=etype.upper(), color=color, density=True)
axes[1].set_xlabel('Event Duration (seconds)')
axes[1].set_ylabel('Density')
axes[1].set_title('Duration by Event Type')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/eda_event_durations.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Duration stats:')
print(dur_df.groupby('type')['duration'].describe().round(1))

## Summary

Key findings from EDA:
1. All 50 patients have usable HR + SpO2 data at 1Hz
2. Both hypopnea and OSA events are present -- we treat both as 'apnea'
3. AHI ranges from mild to severe across patients
4. HR missing data varies significantly (some patients have >30% missing)
5. SpO2 desaturations and HR variability show clear differences between apnea and normal windows
6. Class balance varies per patient but overall the dataset is reasonably balanced for binary classification